In [ ]:
import equinox as eqx
import grain
import jax
import jax.numpy as jnp
import optax
from context_flux_no.data import TheWellDataSource
from context_flux_no.models.multiphysics import (
    AbstractMultiphysicsOperator,
    HyperFluxFNO,
)
from context_flux_no.training.trainer import Trainer
from jaxtyping import Array, Float, PRNGKeyArray


jax.config.update("jax_default_device", jax.devices("gpu")[5])

In [2]:
source = TheWellDataSource(
    "data/datasets",
    "euler_multi_quadrants_periodicBC",
    window_size=11,
    downsample_spatial=8,
    exclude_field_names=["pressure"],
)
sampler = grain.samplers.IndexSampler(len(source), shuffle=True, seed=0)
transforms = [grain.transforms.Batch(batch_size=128)]

In [ ]:
source.metadata_common["spatial_resolution"]

(np.float32(0.015655577), np.float32(0.015655577))

In [ ]:
dataloader = grain.DataLoader(
    data_source=source,
    sampler=sampler,
    operations=transforms,
    worker_count=8,
    # read_options=grain.ReadOptions(num_threads=32, prefetch_buffer_size=1000),
)

In [4]:
dataiter = iter(dataloader)

In [5]:
batch = next(dataiter)

In [5]:
fluxfno = HyperFluxFNO(
    num_spatial_dims=2,
    in_channels=4,
    in_timesteps=None,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(64, 64),
        patch_size=(8, 8),
        depth=4,
        temporal_block_width=128,
        num_heads=4,
        mlp_hidden_dim=128,
    ),
    depth=4,
    frequency_modes=8,
    lift_dim=32,
    stencil_size=(4, 4),
    blocks_hyper=8,
    key=jax.random.key(0),
)

/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/nn/structured_linear.py:40: UserWarning: out_features is not divisible by num_blocks. Output vector 
            will be truncated to the requested size.
  warnings.warn("""out_features is not divisible by num_blocks. Output vector


In [7]:
fluxfno.num_parameters()

19241680

In [6]:
fluxfno = HyperFluxFNO(
    num_spatial_dims=2,
    in_channels=4,
    in_timesteps=10,
    embedding_dim=128,
    encoder_type="DPOT",
    encoder_kwargs=dict(
        grid_size=(64, 64),
        patch_size=(8, 8),
        max_frequency_modes=8,
        fno_depth=4,
        num_blocks=8,
        hidden_dim_patch=128,
        hidden_dim_fno=128,
    ),
    depth=4,
    frequency_modes=8,
    lift_dim=32,
    stencil_size=(4, 4),
    key=jax.random.key(0),
)

In [14]:
print(source.metadata_common["temporal_resolution"])

1.0


In [4]:
def get_spacetime_resolution(data_source: TheWellDataSource):
    time_res = data_source.metadata_common["temporal_resolution"]
    space_res = data_source.metadata_common["spatial_resolution"]
    return tuple([time_res, *space_res])

In [13]:
(batch.itemsize * batch.size) / 1e9

0.092274688

In [8]:
fluxfno(batch[0, :-1], (0.015, 1 / 64, 1 / 64))

E0406 20:57:26.049709 3766217 xtile_compiler.cc:399] Fusion: gemm_fusion_dot = f32[128,640]{1,0} fusion(a.1, bitcast.14), kind=kCustom, calls=gemm_fusion_dot_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0406 20:57:26.049787 3766217 xtile_compiler.cc:401] Computation: gemm_fusion_dot_computation.clone {
  parameter_0 = f32[128,128]{1,0} parameter(0)
  parameter_1 = f32[128,640]{0,1} parameter(1)
  ROOT dot.1 = f32[128,640]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["64"]}
}


(Array([[[ 1.4629129 ,  1.3324407 ,  1.7336452 , ...,  1.0250353 ,
           1.1316228 ,  1.6371596 ],
         [ 1.6007245 ,  1.4711034 ,  1.5532452 , ...,  1.7078118 ,
           1.187907  ,  1.249449  ],
         [ 0.96841633,  1.4407983 ,  1.5290021 , ...,  1.225656  ,
           1.9302391 ,  1.8678191 ],
         ...,
         [ 2.418906  ,  2.5003316 ,  1.9764601 , ...,  1.9809244 ,
           1.6885366 ,  2.1053925 ],
         [ 2.1882892 ,  2.475337  ,  1.5681692 , ...,  1.5525868 ,
           1.949562  ,  2.359899  ],
         [ 1.4100926 ,  1.6261834 ,  1.4282436 , ...,  1.6076986 ,
           0.94937176,  1.5618514 ]],
 
        [[ 1.1998624 ,  1.3571541 ,  1.729555  , ...,  0.80995774,
           0.97408706,  1.4797004 ],
         [ 1.5086582 ,  1.4018352 ,  1.8748505 , ...,  1.2366563 ,
           1.3641735 ,  1.9158852 ],
         [ 1.5790395 ,  1.5289398 ,  1.5177634 , ...,  1.1019827 ,
           1.6830921 ,  1.8118896 ],
         ...,
         [ 2.2602658 ,  2.3130708

In [9]:
%%timeit
eqx.filter_jit(eqx.filter_vmap(fluxfno))(batch[:, :-1], (1.0, 1.0, 1.0))[
    0
].block_until_ready()

E0406 20:57:46.925519 3766186 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.73 = f32[128,128]{1,0} fusion(mul.461, dynamic_nodonate__fun___139_.1), kind=kCustom, calls=gemm_fusion_dot.73_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0406 20:57:46.925675 3766186 xtile_compiler.cc:401] Computation: gemm_fusion_dot.73_computation.clone {
  parameter_0.53 = f32[128,128]{1,0} parameter(0)
  parameter_1.53 = f32[128,128]{1,0} parameter(1)
  ROOT dot.167 = f32[128,128]{1,0} dot(parameter_0.53, parameter_1.53), lhs_contracting_dims={1}, rhs_contracting_dims={1}, backend_config={"sizes":["64"]}
}
E0406 20:57:46.970982 376624

74.1 ms ± 16.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
fluxfno.num_parameters()

140948936

In [6]:
def loss_fn(
    model: AbstractMultiphysicsOperator,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


trainer = Trainer(
    optax.adamw(1e-3),
    loss_fn,
    "./",
    None,
    {"entity": "jhko725", "project": "euler_test"},
    config_dict={"model": ""},
)

In [7]:
trainer.train(
    fluxfno, dataloader, None, loss_args=(0.015, 1 / 64, 1 / 64), num_steps=500, seed=0
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jhko725/.netrc.
wandb: Currently logged in as: jhko725 (jhelab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


E0406 21:07:11.416542 3791432 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.435 = f32[128,128]{0,1} fusion(add_any.401, reduce_sum.2715), kind=kCustom, calls=gemm_fusion_dot_general.435_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0406 21:07:11.416704 3791432 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.435_computation.clone {
  parameter_1.207 = f32[128,128]{1,0} parameter(1)
  constant_1785 = f32[] constant(0.015625)
  broadcast.1678 = f32[128,128]{1,0} broadcast(constant_1785), dimensions={}
  multiply.951 = f32[128,128]{1,0} multiply(parameter_1.207, broadcast.1678)
  parameter_0.207 = f3

Step: 1 | Train loss: 3.2435832023620605 | Valid loss: 
            None
Step: 2 | Train loss: 0.20329013466835022 | Valid loss: 
            None


Step: 3 | Train loss: 0.21688686311244965 | Valid loss: 
            None


Step: 4 | Train loss: 0.29175087809562683 | Valid loss: 
            None


...

Step: 204 | Train loss: 0.048361048102378845 | Valid loss: 
            None


Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
Process grain-process-prefetch-ProcessPrefetchDatasetIterator(buffer_size=1):
ERROR:absl:[process=0][thread=MainThread][step=204][wait_until_f

ConnectionResetError: Connection lost

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x150f626beae0>> (for post_run_cell), with arguments args (<ExecutionResult object at 150e4c1a2ab0, execution_count=7 error_before_exec=None error_in_exec=Connection lost info=<ExecutionInfo object at 150e4c1a2510, raw_cell="trainer.train(
    fluxfno, dataloader, None, loss.." transformed_cell="trainer.train(
    fluxfno, dataloader, None, loss.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bkias-syntax-compute/home/jhko725/projects/CONTEXT_FLUX_NO/prototype_encoders.ipynb#X22sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost